# Feature Engineering Check: dev_profile_final_v4

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_DIR = "Data"
SAMPLE_PARQUET = DATA_DIR + "/dev_profile_sample.parquet"
DB_PATH = "developer_project.duckdb"
TABLE = "dev_profile_final_v4"

import os
con = duckdb.connect(":memory:")

if os.path.exists(SAMPLE_PARQUET):
    con.execute("CREATE TABLE " + TABLE + " AS SELECT * FROM read_parquet(" + repr(SAMPLE_PARQUET) + ")")
    print("Loaded from:", SAMPLE_PARQUET)
elif os.path.exists(DB_PATH):
    src = duckdb.connect(DB_PATH, read_only=True)
    tables_in_db = {r[0] for r in src.execute("SHOW TABLES").fetchall()}
    if TABLE not in tables_in_db:
        src.close()
        raise RuntimeError(TABLE + " not in " + DB_PATH + ". Run FeatureEngineering_Sample.ipynb first.")
    df = src.execute("SELECT * FROM " + TABLE).fetchdf()
    src.close()
    con.register("_df", df)
    con.execute("CREATE TABLE " + TABLE + " AS SELECT * FROM _df")
    con.unregister("_df")
    print("Loaded from:", DB_PATH)
else:
    raise FileNotFoundError("Run FeatureEngineering_Sample.ipynb first to generate " + SAMPLE_PARQUET)

n = con.execute("SELECT COUNT(*) FROM " + TABLE).fetchone()[0]
print(TABLE + ":", n, "rows")


## 0. Load data into pandas

In [ ]:
import seaborn as sns
import numpy as np
sns.set_theme(style="whitegrid", palette="muted")

df = con.execute("SELECT * FROM " + TABLE).fetchdf()
print("Shape:", df.shape)

num_cols  = df.select_dtypes("number").columns.tolist()
cat_cols  = df.select_dtypes("object").columns.tolist()
bool_cols = [c for c in num_cols if set(df[c].dropna().unique()).issubset({0, 1})]
print("Numeric:", len(num_cols), " Categorical:", len(cat_cols))

display(df.describe(include="number").T.round(4))


## 1. Data Quality Scorecard

In [ ]:
checks = {}

total = len(df)

checks["unique_developer_id"]          = int(df["developer_id"].nunique() == total)
checks["no_null_developer_id"]         = int(df["developer_id"].notna().all())
checks["is_activated_binary"]          = int(df["is_activated"].isin([0,1]).all())
checks["effort_rank_in_0_4"]           = int(df["developer_effort_rank"].between(0,4).all())
checks["effort_score_non_negative"]    = int((df["developer_effort_score"] >= 0).all())
checks["persona_confidence_0_1"]       = int(df["persona_confidence"].between(0,1).all())
checks["persona_entropy_0_1"]          = int(df["persona_entropy"].between(0,1).all())
checks["lifetime_activity_non_neg"]    = int((df["lifetime_activity_count"] >= 0).all())
checks["lifetime_score_non_neg"]       = int((df["lifetime_activity_score_sum"] >= 0).all())
checks["activity_0_30d_non_neg"]       = int((df["activity_count_0_30d"] >= 0).all())
checks["build_share_0_30d_0_1"]        = int(df["build_share_0_30d"].between(0,1).all())
checks["heff_share_0_30d_0_1"]         = int(df["high_effort_share_0_30d"].between(0,1).all())
checks["act_flag_vs_count_0_30d"]      = int(((df["activity_count_0_30d"]>0)==(df["has_activity_0_30d"]==1)).all())
checks["recent_build_flag_vs_count"]   = int(~((df["recent_build_flag"]==1)&(df["build_count_0_30d"]==0)).any())
checks["no_null_lifecycle_status"]     = int(df["final_lifecycle_status"].notna().all())
checks["no_null_dormancy_status"]      = int(df["dormancy_status"].notna().all())

qa = (
    __import__("pandas").DataFrame(
        {"check": list(checks.keys()), "pass": list(checks.values())}
    )
    .assign(status=lambda d: d["pass"].map({1: "PASS", 0: "FAIL"}))
)
total_pass = qa["pass"].sum()
print(f"QA: {total_pass}/{len(qa)} checks passed")
display(qa.style.applymap(lambda v: "background-color:#c8e6c9" if v=="PASS" else "background-color:#ffcdd2", subset=["status"]))


## 2. Missing Value Audit

In [ ]:
null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
with_nulls = null_pct[null_pct > 0]

print(f"Columns with nulls: {len(with_nulls)} / {len(df.columns)}")
if len(with_nulls):
    display(with_nulls.rename("null_pct").reset_index().rename(columns={"index":"column"}).round(2))

    fig, ax = plt.subplots(figsize=(10, max(3, len(with_nulls)*0.35)))
    with_nulls.sort_values().plot.barh(ax=ax, color="steelblue", edgecolor="k", linewidth=0.3)
    ax.set_xlabel("% null")
    ax.set_title("Missing Value Rate by Column")
    plt.tight_layout()
    plt.show()
else:
    print("No null values found.")


## 3. Persona Analysis

In [ ]:
# Distribution
persona_dist = (
    df.groupby("persona")
    .size()
    .reset_index(name="developers")
    .assign(pct=lambda d: (d["developers"]/len(df)*100).round(2))
    .sort_values("developers", ascending=False)
)
display(persona_dist)

# Stacked bar: persona x lifecycle status
pivot = (
    df.groupby(["final_lifecycle_status","persona"])
    .size()
    .unstack(fill_value=0)
)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

ax = pivot_pct.plot(kind="bar", stacked=True, figsize=(11,5), colormap="tab10", edgecolor="k", linewidth=0.3)
ax.set_ylabel("% of lifecycle status")
ax.set_title("Persona composition by Lifecycle Status")
ax.legend(bbox_to_anchor=(1.01,1), loc="upper left", title="Persona")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Persona share histograms
share_cols = ["cuda_share","genai_share","robotics_share",
              "simulation_share","learning_community_share"]

fig, axes = plt.subplots(1, len(share_cols), figsize=(16,4))
for ax, col in zip(axes, share_cols):
    sns.histplot(df[col].dropna(), bins=20, kde=True, ax=ax, color="steelblue")
    ax.set_title(col.replace("_share",""), fontsize=9)
    ax.set_xlabel("share value")
plt.suptitle("Persona Share Distributions", fontsize=12)
plt.tight_layout()
plt.show()

# Persona confidence vs entropy scatter (sample up to 2000 pts)
sample = df[["persona_confidence","persona_entropy","persona"]].dropna().sample(min(2000,len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(7,5))
for persona, grp in sample.groupby("persona"):
    ax.scatter(grp["persona_confidence"], grp["persona_entropy"], label=persona, alpha=0.4, s=15)
ax.set_xlabel("persona_confidence")
ax.set_ylabel("persona_entropy")
ax.set_title("Persona Confidence vs Entropy")
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plt.show()


## 4. Effort Features

In [ ]:
effort_dist = (
    df.groupby(["developer_effort_level","developer_effort_rank"])
    .agg(developers=("developer_id","count"),
         avg_score=("developer_effort_score","mean"),
         med_score=("developer_effort_score","median"))
    .reset_index()
    .sort_values("developer_effort_rank", ascending=False)
    .round(4)
)
display(effort_dist)

fig, axes = plt.subplots(1, 2, figsize=(13,4))

sns.histplot(df["developer_effort_score"].dropna(), bins=40, kde=True, ax=axes[0], color="darkorange")
axes[0].set_title("developer_effort_score distribution")
axes[0].set_xlabel("score")

order = ["Passive","Low","Moderate","High","Very High"]
present = [l for l in order if l in df["developer_effort_level"].unique()]
sns.boxplot(data=df[df["developer_effort_level"].notna()],
            x="developer_effort_level", y="developer_effort_score",
            order=present, ax=axes[1], palette="OrRd", linecolor="black")
axes[1].set_title("Effort Score by Effort Level")
axes[1].set_xlabel("")
plt.tight_layout()
plt.show()


In [ ]:
# Effort score by lifecycle status
lc_order = df.groupby("final_lifecycle_status")["developer_id"].count().sort_values(ascending=False).index.tolist()
fig, ax = plt.subplots(figsize=(11,4))
sns.boxplot(data=df[df["developer_effort_score"].notna()],
            x="final_lifecycle_status", y="developer_effort_score",
            order=lc_order, ax=ax, palette="coolwarm", linecolor="black")
ax.set_title("Effort Score by Lifecycle Status")
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 5. Journey Stage & Dormancy

In [ ]:
for col in ["behavior_journey_stage_30d","current_journey_state_30d",
             "dormancy_status","final_lifecycle_status"]:
    dist = (
        df.groupby(col, dropna=False)
        .size()
        .reset_index(name="n")
        .assign(pct=lambda d: (d["n"]/len(df)*100).round(2))
        .sort_values("n", ascending=False)
    )
    print("===", col, "===")
    display(dist)


In [ ]:
# Cross-tab heatmap: dormancy_status x final_lifecycle_status
ct = (
    df.groupby(["dormancy_status","final_lifecycle_status"])
    .size()
    .unstack(fill_value=0)
)
fig, ax = plt.subplots(figsize=(max(8,ct.shape[1]*1.4), max(4,ct.shape[0]*0.8)))
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", linewidths=0.5, ax=ax)
ax.set_title("Dormancy Status x Lifecycle Status (counts)")
ax.set_ylabel("dormancy_status")
ax.set_xlabel("final_lifecycle_status")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# Binary flag validity
flag_cols = ["is_activated","dormant_flag","at_risk_flag","cooling_flag","missing_contact_metadata_flag"]
flag_valid = {c: df[c].isin([0,1]).all() for c in flag_cols if c in df.columns}
print("Binary flag validity:", flag_valid)


## 6. Recency Window Features

In [ ]:
windows = [("0_30d",0,30),("30_90d",30,90),("90_180d",90,180)]
cols_to_check = [f"activity_count_{w}" for w,*_ in windows] +                 [f"build_count_{w}" for w,*_ in windows] +                 [f"high_effort_count_{w}" for w,*_ in [("0_30d",0,30),("30_90d",30,90)]]
existing = [c for c in cols_to_check if c in df.columns]
display(df[existing].describe().T.round(2))

# Pct with any activity per window
for w,*_ in windows:
    c = f"activity_count_{w}"
    if c in df.columns:
        pct = (df[c]>0).mean()*100
        print(f"  {w}: {pct:.1f}% of developers have any activity")


In [ ]:
# Histplots: activity count per window (active devs only, log scale)
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax, (w,*_) in zip(axes, windows):
    c = f"activity_count_{w}"
    if c not in df.columns:
        ax.set_visible(False); continue
    active = df[df[c]>0][c].clip(upper=df[c].quantile(0.99))
    sns.histplot(active, bins=30, kde=True, ax=ax, color="teal")
    ax.set_title(f"activity_count_{w}")
    ax.set_xlabel("count (capped p99)")
plt.suptitle("Activity Counts (active devs only)", fontsize=12)
plt.tight_layout()
plt.show()

# Build share vs effort share scatter (0-30d)
if "build_share_0_30d" in df.columns and "high_effort_share_0_30d" in df.columns:
    sample = df[["build_share_0_30d","high_effort_share_0_30d","developer_effort_level"]].dropna().sample(min(2000,len(df)),random_state=1)
    fig, ax = plt.subplots(figsize=(6,5))
    for lvl, grp in sample.groupby("developer_effort_level"):
        ax.scatter(grp["build_share_0_30d"], grp["high_effort_share_0_30d"],
                   label=lvl, alpha=0.4, s=15)
    ax.set_xlabel("build_share_0_30d")
    ax.set_ylabel("high_effort_share_0_30d")
    ax.set_title("Build Share vs High-Effort Share (0-30d)")
    ax.legend(markerscale=2, fontsize=8)
    plt.tight_layout()
    plt.show()


## 7. Lifetime Features

In [ ]:
lt_cols = ["lifetime_activity_count","lifetime_activity_score_sum",
           "lifetime_unique_activity_types","lifetime_unique_modalities",
           "lifetime_active_weeks","lifetime_build_count",
           "lifetime_champion_count","lifetime_high_effort_count",
           "lifetime_dli_training_count","lifetime_avg_effort_rank","lifetime_max_effort_rank"]
lt_cols = [c for c in lt_cols if c in df.columns]

# Percentile table
pcts = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
pct_df = df[lt_cols].quantile(pcts).T
pct_df.columns = [f"p{int(p*100)}" for p in pcts]
pct_df.insert(0, "mean", df[lt_cols].mean().round(2))
pct_df.insert(0, "min",  df[lt_cols].min())
pct_df["max"] = df[lt_cols].max()
display(pct_df.round(2))


In [ ]:
# Log-scale histplots for skewed lifetime counts
plot_lt = [c for c in ["lifetime_activity_count","lifetime_activity_score_sum",
                       "lifetime_active_weeks","lifetime_high_effort_count",
                       "lifetime_build_count","lifetime_unique_activity_types"]
           if c in df.columns]
fig, axes = plt.subplots(2, 3, figsize=(15,8))
axes = axes.flatten()
for i, col in enumerate(plot_lt):
    active = df[df[col]>0][col]
    sns.histplot(active, bins=40, kde=True, ax=axes[i], log_scale=(False,True), color="indigo")
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("value")
    axes[i].set_ylabel("count (log)")
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Lifetime Feature Distributions (log y-axis, active devs only)", fontsize=11)
plt.tight_layout()
plt.show()


## 8. Numeric Correlation Heatmap

In [ ]:
key_numeric = [c for c in [
    "lifetime_activity_count","lifetime_activity_score_sum",
    "lifetime_unique_activity_types","lifetime_active_weeks",
    "lifetime_build_count","lifetime_high_effort_count",
    "activity_count_0_30d","build_count_0_30d","high_effort_count_0_30d",
    "build_share_0_30d","high_effort_share_0_30d",
    "developer_effort_score","developer_effort_rank",
    "lifetime_avg_effort_rank","weighted_recent_activity",
    "activity_velocity_0_30_vs_30_90",
    "persona_confidence","persona_entropy",
    "cuda_share","genai_share","is_activated",
] if c in df.columns]

corr = df[key_numeric].corr()

fig, ax = plt.subplots(figsize=(14,11))
mask = __import__("numpy").triu(__import__("numpy").ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.4,
            annot_kws={"size":7}, ax=ax)
ax.set_title("Pearson Correlation - Key Numeric Features", fontsize=13)
plt.xticks(rotation=40, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()


## 9. Grouped Analysis

In [ ]:
# Summary stats by lifecycle status
summary = (
    df.groupby("final_lifecycle_status")
    .agg(
        developers=("developer_id","count"),
        pct_activated=("is_activated","mean"),
        avg_lt_activity=("lifetime_activity_count","mean"),
        med_lt_activity=("lifetime_activity_count","median"),
        avg_effort_score=("developer_effort_score","mean"),
        avg_act_0_30d=("activity_count_0_30d","mean"),
        avg_lt_types=("lifetime_unique_activity_types","mean"),
    )
    .assign(pct_activated=lambda d: (d["pct_activated"]*100).round(1))
    .round(2)
    .sort_values("developers", ascending=False)
)
display(summary)

# Box plots: lifetime_activity_count by lifecycle status
lc_order = summary.index.tolist()
fig, axes = plt.subplots(1,2, figsize=(14,5))
for ax, col, title in [
    (axes[0], "lifetime_activity_count",   "Lifetime Activity Count"),
    (axes[1], "developer_effort_score",    "Effort Score"),
]:
    valid = df[df[col].notna()]
    sns.boxplot(data=valid, x="final_lifecycle_status", y=col,
                order=[s for s in lc_order if s in valid["final_lifecycle_status"].unique()],
                ax=ax, palette="Set2", showfliers=False, linecolor="black")
    ax.set_title(title)
    ax.set_xlabel("")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Activation rate by account_type and region
for group_col in ["account_type","region"]:
    if group_col not in df.columns:
        continue
    act_rate = (
        df.groupby(group_col)
        .agg(developers=("developer_id","count"),
             activated=("is_activated","sum"))
        .assign(activation_pct=lambda d: (d["activated"]/d["developers"]*100).round(1))
        .sort_values("developers", ascending=False)
        .head(15)
    )
    print("=== Activation rate by", group_col, "===")
    display(act_rate)

    fig, ax = plt.subplots(figsize=(10,4))
    bars = ax.bar(act_rate.index.astype(str), act_rate["activation_pct"], color="steelblue", edgecolor="k", linewidth=0.3)
    ax2 = ax.twinx()
    ax2.plot(range(len(act_rate)), act_rate["developers"], color="darkorange", marker="o", linewidth=1.5, label="developers")
    ax.set_ylabel("activation %")
    ax2.set_ylabel("total developers", color="darkorange")
    ax.set_title("Activation Rate by " + group_col)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


## 10. Outlier Audit

In [ ]:
# IQR-based outlier count per numeric column
iqr_cols = [c for c in [
    "lifetime_activity_count","lifetime_activity_score_sum",
    "activity_count_0_30d","developer_effort_score",
    "weighted_recent_activity","activity_velocity_0_30_vs_30_90",
] if c in df.columns]

iqr_rows = []
for col in iqr_cols:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_out = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()
    iqr_rows.append({"column": col, "q1": round(q1,2), "q3": round(q3,2),
                     "iqr": round(iqr,2), "outliers_iqr": n_out,
                     "outlier_pct": round(n_out/len(df)*100,2)})
display(__import__("pandas").DataFrame(iqr_rows))


In [ ]:
# Top 10 developers by lifetime_activity_count
display(
    df.nlargest(10, "lifetime_activity_count")[[
        "developer_id","lifetime_activity_count","lifetime_activity_score_sum",
        "lifetime_unique_activity_types","developer_effort_level",
        "final_lifecycle_status","persona","country"
    ]]
)


## 11. Cross-Feature Consistency

In [ ]:
consistency = {}

if "activity_count_0_30d" in df.columns:
    consistency["act_flag_match_0_30d"]   = int(~((df["activity_count_0_30d"]>0)&(df.get("has_activity_0_30d",__import__("pandas").Series(dtype=int))==0)).any())
if "activity_count_30_90d" in df.columns:
    consistency["act_flag_match_30_90d"]  = int(~((df["activity_count_30_90d"]>0)&(df.get("has_activity_30_90d",__import__("pandas").Series(dtype=int))==0)).any())
if "build_count_0_30d" in df.columns and "recent_build_flag" in df.columns:
    consistency["recent_build_flag_ok"]   = int(~((df["recent_build_flag"]==1)&(df["build_count_0_30d"]==0)).any())
if "is_activated" in df.columns:
    consistency["unactivated_no_status_mismatch"] = int(~((df["final_lifecycle_status"]=="Unactivated")&(df["lifetime_activity_count"]>0)&(df["is_activated"]==1)).any())
if "lifetime_avg_effort_rank" in df.columns:
    consistency["avg_le_max_effort_rank"] = int((df["lifetime_avg_effort_rank"] <= df["lifetime_max_effort_rank"]).all())
if "persona_entropy" in df.columns:
    consistency["entropy_non_negative"]   = int((df["persona_entropy"] >= -1e-6).all())

result = __import__("pandas").DataFrame(
    {"check": list(consistency.keys()), "pass": list(consistency.values())}
).assign(status=lambda d: d["pass"].map({1:"PASS",0:"FAIL"}))
display(result.style.applymap(lambda v: "background-color:#c8e6c9" if v=="PASS" else "background-color:#ffcdd2", subset=["status"]))
